In [24]:
# Regularised linear regression models (Ridge, Lasso, ElasticNet)
# used to reduce overfitting and improve generalisation
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Using df from Initial Data Preparation (cleaned, log_streams created, outliers removed)


In [25]:
# 4.1.2 Feature Definition and Target Variable

import pandas as pd
import numpy as np

# 1) Load the cleaned dataset from Part 3
df = pd.read_csv("spotify_initial_prepared.csv")

# 2) Make sure numeric columns are really numeric (remove commas if any)
numeric_cols_for_modelling = [
    "streams",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%",
]

for col in numeric_cols_for_modelling:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3) Ensure log_streams exists – recreate if missing
if "log_streams" not in df.columns:
    df["log_streams"] = np.log1p(df["streams"])
    print("log_streams was missing – recreated from streams.")

# Drop any rows with missing values in important numeric columns or target
df = df.dropna(subset=numeric_cols_for_modelling + ["log_streams"])

print("Shape after numeric cleaning and dropping NaNs:", df.shape)

# 4) Define target and predictors
target_col = "log_streams"

feature_cols = [
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%",
    "key",
    "mode",
]

# 5) Subset to selected features + target and drop rows with missing data
df_model = df[feature_cols + [target_col]].dropna()

# 6) Feature matrix X and target vector y
X = df_model[feature_cols]
y = df_model[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)


Shape after numeric cleaning and dropping NaNs: (900, 25)
X shape: (814, 21)
y shape: (814,)


In [26]:
# 4.1.3 Train–Test Split and Preprocessing Pipeline

# Train–test split (80/20, same random_state as baseline for fair comparison)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# Numeric and categorical predictors
numeric_features = [
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%",
]

categorical_features = ["key", "mode"]

# ColumnTransformer: one-hot encode categoricals, pass numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)


Train shape: (651, 21) (651,)
Test shape: (163, 21) (163,)


In [27]:
# 4.1.4 Hyper-parameters and Model Specifications

from sklearn.preprocessing import StandardScaler

# Pipeline builder: preprocessor + scaler + model
def make_pipeline_with_model(model):
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("scaler", StandardScaler(with_mean=False)),  # sparse-safe
            ("model", model),
        ]
    )

# Hyper-parameter grids

ridge_param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}

lasso_param_grid = {
    "model__alpha": [0.0005, 0.001, 0.01, 0.1, 1.0]
}

elasticnet_param_grid = {
    "model__alpha": [0.0005, 0.001, 0.01, 0.1, 1.0],
    "model__l1_ratio": [0.2, 0.5, 0.8],
}


In [28]:
# 4.1.5 Model Fitting, Hyper-parameter Tuning and Evaluation

import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

def tune_model(name, base_estimator, param_grid):
    # Build the full pipeline: preprocessor + scaler + model
    pipe = make_pipeline_with_model(base_estimator)

    # GridSearchCV with 5-fold cross-validation
    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="neg_root_mean_squared_error",  # still ok to use
        n_jobs=-1,
        verbose=1,
    )

    # Fit on training data
    grid_search.fit(X_train, y_train)

    # Best model from the search
    best_model = grid_search.best_estimator_

    # Predictions on train and test
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    # Compute MSE then RMSE manually (because 'squared' argument is not supported)
    mse_train = mean_squared_error(y_train, y_train_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)
    rmse_train = np.sqrt(mse_train)
    rmse_test = np.sqrt(mse_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    metrics = {
        "Train_RMSE": rmse_train,
        "Test_RMSE": rmse_test,
        "Train_R2": r2_train,
        "Test_R2": r2_test,
    }

    print(f"\n{name} - Best hyper-parameters: {grid_search.best_params_}")
    print(f"{name} - Train RMSE: {rmse_train:.4f}, Test RMSE: {rmse_test:.4f}")
    print(f"{name} - Train R²:   {r2_train:.4f}, Test R²:   {r2_test:.4f}")

    return best_model, metrics


# Tune three models again with corrected RMSE calculation

ridge_best, ridge_metrics = tune_model(
    "Ridge",
    Ridge(random_state=42),
    ridge_param_grid
)

lasso_best, lasso_metrics = tune_model(
    "Lasso",
    Lasso(random_state=42, max_iter=10000),
    lasso_param_grid
)

elasticnet_best, elasticnet_metrics = tune_model(
    "ElasticNet",
    ElasticNet(random_state=42, max_iter=10000),
    elasticnet_param_grid
)


Fitting 5 folds for each of 4 candidates, totalling 20 fits

Ridge - Best hyper-parameters: {'model__alpha': 10.0}
Ridge - Train RMSE: 0.7101, Test RMSE: 0.6766
Ridge - Train R²:   0.5496, Test R²:   0.5068
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Lasso - Best hyper-parameters: {'model__alpha': 0.01}
Lasso - Train RMSE: 0.7126, Test RMSE: 0.6734
Lasso - Train R²:   0.5465, Test R²:   0.5114
Fitting 5 folds for each of 15 candidates, totalling 75 fits

ElasticNet - Best hyper-parameters: {'model__alpha': 0.1, 'model__l1_ratio': 0.2}
ElasticNet - Train RMSE: 0.7227, Test RMSE: 0.6696
ElasticNet - Train R²:   0.5335, Test R²:   0.5168


In [30]:
# 4.1.6 Summary of Model Performance

results_regularised = pd.DataFrame([
    {
        "Model": "Ridge (tuned)",
        "Train_RMSE": ridge_metrics["Train_RMSE"],
        "Test_RMSE": ridge_metrics["Test_RMSE"],
        "Train_R2": ridge_metrics["Train_R2"],
        "Test_R2": ridge_metrics["Test_R2"],
    },
    {
        "Model": "Lasso (tuned)",
        "Train_RMSE": lasso_metrics["Train_RMSE"],
        "Test_RMSE": lasso_metrics["Test_RMSE"],
        "Train_R2": lasso_metrics["Train_R2"],
        "Test_R2": lasso_metrics["Test_R2"],
    },
    {
        "Model": "ElasticNet (tuned)",
        "Train_RMSE": elasticnet_metrics["Train_RMSE"],
        "Test_RMSE": elasticnet_metrics["Test_RMSE"],
        "Train_R2": elasticnet_metrics["Train_R2"],
        "Test_R2": elasticnet_metrics["Test_R2"],
    },
])

print("\nRegularised Linear Models – Performance Summary:")
print(results_regularised)



Regularised Linear Models – Performance Summary:
                Model  Train_RMSE  Test_RMSE  Train_R2   Test_R2
0       Ridge (tuned)    0.710114   0.676558  0.549613  0.506769
1       Lasso (tuned)    0.712600   0.673407  0.546455  0.511353
2  ElasticNet (tuned)    0.722697   0.669623  0.533511  0.516829
